# Resume Detector - Kelompok 5

### Library
---

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [ ]:
import os
import time
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sentence_transformers import SentenceTransformer, InputExample, losses, util
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
from torch.utils.data import DataLoader

### Import Data
---

In [ ]:
DATASET_PATH = "Resume Dataset\Resume\Resume.csv"
def load_and_prepare_data(DATASET_PATH):
    print("[-] Load Data")
    df = pd.read_csv(DATASET_PATH)
    
    df = df.dropna(subset=['Resume_str', 'Category'])
    return df

### Preprocessing
---

In [ ]:
def preprocess_data(df):
    print("[-] Preprocessing Data")
    
    df['cleaned_resume'] = df['Resume_str'].apply(lambda x: " ".join(str(x).split()))
    df['cleaned_category'] = df['Category'].apply(lambda x: str(x).strip())
    
    examples = []
    for _, row in df.iterrows():
        examples.append(InputExample(
            texts=[row['cleaned_resume'], row['cleaned_category']], 
            label=1.0
        ))
        
    return df, examples

In [ ]:
if __name__ == "__main__":
    MODEL_NAME = "all-MiniLM-L6-V2"
    OUTPUT_DIR = "./exported_resume_transformer"

    if not os.path.exists(DATASET_PATH):
        raise FileNotFoundError(f"Please place the dataset at: {DATASET_PATH}")

    # Load and Clean
    df = load_and_prepare_data(DATASET_PATH)
    df, all_examples = preprocess_data(df)

### Split and Train
---

In [ ]:
print("[-] Splitting dataset into Train and Validation")

train_examples, val_examples = train_test_split(all_examples, test_size=0.15, random_state=42)
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)

### Modelling (all-MiniLM-L6-V2)
---

In [ ]:
print(f"[-] Initializing base model: {MODEL_NAME}")

model = SentenceTransformer(MODEL_NAME)

train_loss = losses.CosineSimilarityLoss(model)

### Evaluation
---

In [ ]:
class Evaluator:
    def __init__(self, model):
        self.model = model
        
    def calculate_metrics(self, test_df):
        print("\n[-] RUNNING NLP METRICS EVALUATION ")
        
        resume_col = 'Resume_str' if 'Resume_str' in test_df.columns else 'resume_str'
        category_col = 'Category' if 'Category' in test_df.columns else 'category'
        
        resumes = test_df[resume_col].tolist()
        categories = test_df[category_col].tolist()
        unique_categories = sorted(list(set(categories)))

        start_time = time.time()
        resume_embeddings = self.model.encode(resumes, convert_to_tensor=True, show_progress_bar=False)
        category_embeddings = self.model.encode(unique_categories, convert_to_tensor=True, show_progress_bar=False)
        avg_speed = (time.time() - start_time) / len(resumes)
        
        similarity_matrix = util.cos_sim(resume_embeddings, category_embeddings).cpu().numpy()
        
        predictions = []
        target_scores = []
        unrelated_scores = []
        ndcg_scores = []
        
        for idx, actual_cat in enumerate(categories):
            scores = similarity_matrix[idx]
            
            best_match_idx = np.argmax(scores)
            predictions.append(unique_categories[best_match_idx])
            
            target_idx = unique_categories.index(actual_cat)
            target_scores.append(scores[target_idx])
            
            unrelated_idx = best_match_idx if unique_categories[best_match_idx] != actual_cat else (best_match_idx + 1) % len(unique_categories)
            unrelated_scores.append(scores[unrelated_idx])
            
            sorted_indices = np.argsort(scores)[::-1]
            ranked_categories = [unique_categories[i] for i in sorted_indices]
            rank_pos = ranked_categories.index(actual_cat)
            ndcg_scores.append(1.0 / np.log2(rank_pos + 2))

        print("\n" + "="*60)
        print("A. METRIC CLASSIFICATION (Precision, Recall, F1, Accuracy)")
        print("="*60)
        print(classification_report(categories, predictions, target_names=unique_categories))
        
        print("\n CONFUSION MATRIX")
        print(confusion_matrix(categories, predictions, labels=unique_categories))
        
        print("\n" + "="*60)
        print("B. METRIC RANKING AND SIMILARITY")
        print("="*60)
        print(f"Mean Cosine Similarity (Target Field):    {np.mean(target_scores):.4f}")
        print(f"Mean Cosine Similarity (Unrelated Field): {np.mean(unrelated_scores):.4f}")
        print(f"Mean NDCG Top-Rank Score:                {np.mean(ndcg_scores):.4f}")
        print(f"Mean Average Precision (MAP) Estimate:   {np.mean(ndcg_scores):.4f}")
        print(f"Average Processing Speed Per CV:         {avg_speed:.4f} seconds")
        print("="*60 + "\n")

In [ ]:
print("[-] Fine-Tuning started...")

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=4,
    warmup_steps=100,
    output_path=os.path.join(OUTPUT_DIR, "checkpoint")
)

print("[+] Fine-Tuning complete!")

In [ ]:
val_data = []
for example in val_examples:
    val_data.append({
        'Resume_str': example.texts[0],  # The resume text
        'Category': example.texts[1]     # The job category label
    })
evaluation_df = pd.DataFrame(val_data)

evaluator = Evaluator(model)

evaluator.calculate_metrics(evaluation_df)

### Export Model
---

In [ ]:
print(f"[-] Exporting Final Model - {OUTPUT_DIR}")

model.save(OUTPUT_DIR)

print("[+] Model successfully exported! Ready for your demo deployment.")

In [ ]:
class ResumeMatcher:
    def __init__(self, model_path="./exported_resume_transformer"):
        print("[-] Loading exported model into memory...")
        self.model = SentenceTransformer(model_path)
        
    def score_similarity(self, resume_text, job_descriptions):
        """
        Compares an incoming resume string against a list of job criteria or roles.
        Returns a sorted list of matched targets with cosine scores.
        """
        resume_embedding = self.model.encode(resume_text, convert_to_tensor=True)
        job_embeddings = self.model.encode(job_descriptions, convert_to_tensor=True)
        
        cosine_scores = util.cos_sim(resume_embedding, job_embeddings)[0]
        
  
        results = []
        for i, score in enumerate(cosine_scores):
            results.append({
                "job_description": job_descriptions[i],
                "match_score": round(float(score), 4)
            })
            
        
        results = sorted(results, key=lambda x: x['match_score'], reverse=True)
        return results

if __name__ == "__main__":
    matcher = ResumeMatcher()
    
    sample_cv = "Experienced software engineer specializing in Python development, building scalable API endpoints with FastAPI, PostgreSQL configurations, and deploying via Docker containers."
    job_targets = ["Data Scientist Roles", "Fullstack Web Developer", "Human Resources Manager", "DevOps Engineer"]
    
    rankings = matcher.score_similarity(sample_cv, job_targets)
    
    print("\n[+] Match Rankings for Demo :")
    for rank in rankings:
        print(f"Target: {rank['job_description']} | Score: {rank['match_score']}")